# Evaluate a fine-tuned ARDB adapter against real, hand-verified ground truth

Standalone from the training notebooks (`colab_gemma4_e2b_finetune.ipynb` /
`colab_qwen35_finetune.ipynb`) on purpose: every fine-tune ends up as a portable adapter repo on
the Hub regardless of which notebook trained it, so evaluation doesn't need to live inside either
training notebook or be duplicated across both -- this one notebook evaluates **any** pushed ARDB
adapter (Gemma or Qwen) by just changing `_MODEL_FAMILY` / `_ADAPTER_TO_SCORE` below.

Scores against the 5 real ARDB bulletins in `eval/datasets/real` (hand-verified ground truth,
ARDB documents only -- the corpus's non-ARDB test file is excluded), a stronger signal than the
synthetic template-substituted validation split the training notebooks' own eval cells use.

**Steps:**
1. Set `_MODEL_FAMILY` and `_ADAPTER_TO_SCORE` in the cell below, then run the install cell
   (matches whichever training notebook's proven recipe, gated by `_MODEL_FAMILY` -- Gemma and
   Qwen need different, non-overlapping installs, so restart the runtime if you switch families
   in the same session, same as switching configs within either training notebook).
2. In the Files panel (left sidebar), create a folder named `real_docs`, then upload the 15 real
   page PNGs from your local `eval/datasets/real/` (the 5 ARDB documents' pages, not
   `CambodiaBudgetExecutioninApr-2024`).
3. Run the remaining cells. Download `real_predictions.json` when done and score it locally:
   ```bash
   uv run python3 scripts/eval_finetune_real.py --predictions real_predictions.json
   ```

In [ ]:
# Set these two, then run this cell and everything below.
_MODEL_FAMILY = "gemma"  # "gemma" or "qwen"
_ADAPTER_TO_SCORE = "Soxavin/gemma4-e2b-ardb-lora-v5-e3"  # any pushed adapter repo for that family


In [ ]:
%%capture
import os, re, importlib.util

if _MODEL_FAMILY == "gemma":
    # Byte-matched to colab_gemma4_e2b_finetune.ipynb's own install cell -- proven working
    # (Run 4 trained and evaluated cleanly, see eval/gemma_finetune_runs.md).
    os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"
    if "COLAB_" not in "".join(os.environ.keys()):
        !pip install unsloth
    else:
        import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
        xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
        !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
        !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
        !pip install --no-deps --upgrade "torchao>=0.16.0"
    !pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
    !pip install torchcodec
    !pip install --no-deps --upgrade timm  # Gemma 4 vision

elif _MODEL_FAMILY == "qwen":
    # Byte-matched to colab_qwen35_finetune.ipynb's own install cell -- proven working (see
    # docs/PROJECT_LOG.md #2.107 for the causal_conv1d --no-binary fix this depends on).
    !pip install --upgrade -qqq uv
    if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
        try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
        except: _numpy = "numpy"; _pil = "pillow"
        !uv pip install -qqq \
            "torch==2.10.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.34 \
            "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
            "unsloth[base] @ git+https://github.com/unslothai/unsloth"
        !uv pip install -qqq --no-deps "torchcodec>=0.10.0"
    elif importlib.util.find_spec("unsloth") is None:
        !uv pip install -qqq unsloth
    !uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
    !uv pip install transformers==5.2.0
    !uv pip install --no-build-isolation --no-binary causal_conv1d flash-linear-attention "causal_conv1d>=1.6.0"
    import torch
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8:
        !uv pip install --no-deps "apache-tvm-ffi==0.1.9" "tilelang==0.1.8"
    else:
        os.environ["FLA_TILELANG"] = "0"
    !uv pip install --no-deps --upgrade "torchao>=0.16.0"

else:
    raise ValueError(f"_MODEL_FAMILY must be 'gemma' or 'qwen', got {_MODEL_FAMILY!r}")


In [ ]:
# Outside %%capture on purpose -- prints what actually got installed instead of assuming it.
import torch, unsloth, transformers
print(f"model_family={_MODEL_FAMILY} adapter={_ADAPTER_TO_SCORE}")
print(f"torch={torch.__version__} unsloth={unsloth.__version__} transformers={transformers.__version__}")

## Load adapter

In [ ]:
from unsloth import FastVisionModel

# Loads the adapter + its base together from the Hub -- correct standalone, no dependency on
# any other cell/notebook having run first.
model, processor = FastVisionModel.from_pretrained(_ADAPTER_TO_SCORE, load_in_4bit=False)
FastVisionModel.for_inference(model)
print(f"Loaded {_ADAPTER_TO_SCORE}")


## Score against real ARDB documents

Upload the 15 real page PNGs into a `real_docs` folder (Files panel) before running this.

In [ ]:
from pathlib import Path
from PIL import Image
import json, time

# Copied verbatim from build_ardb_unified_sft.py's _UNIFIED_INSTRUCTION -- fixed across every
# row of every dataset version this task trains on, so hardcoding it here (rather than reading
# it off a loaded dataset) keeps this notebook fully self-contained.
_instruction = (
    "Extract every layout region on this page. Output a JSON list of objects, each "
    '{"box_2d": [y1, x1, y2, x2], "label": category, "text": ...} (text is the region\'s '
    "transcribed content, or an empty string if it has none, e.g. a photo), with box_2d "
    "normalized to a 0-1000 grid. Categories: Table, Text, Section-Header, Page-Furniture, "
    "Picture."
)

# Real pages (unlike the synthetic val split) sometimes trigger runaway repetition -- verbose,
# hallucinated content that never hits EOS, burning the full max_new_tokens every time (this is
# what made the first real-doc run take ~5 min/page). Two mitigations, both logged per page so a
# slow/truncated page is visible in the output rather than silently eating wall-clock or getting
# cut off unnoticed:
#   - no_repeat_ngram_size=4 blocks the model from repeating any 4-token sequence, which breaks
#     the specific repeat-the-same-wrong-row-forever loops seen in earlier hallucinated output
#     (see eval/qwen_finetune_runs.md's Run 1 "Read" section) without constraining legitimately
#     varied long output.
#   - max_new_tokens trimmed 5000 -> 4096: still well above this project's longest known training
#     target (~4000 chars / ~2048 tokens at Qwen's real ceiling, see docs/PROJECT_LOG.md #2.107),
#     so it shouldn't truncate genuine content -- if `hit_cap` ever prints True below, that page's
#     prediction may be truncated and should be treated with caution when scoring.
_MAX_NEW_TOKENS = 4096

_real_dir = Path("real_docs")
_png_paths = sorted(_real_dir.glob("*.png"))
print(f"Found {len(_png_paths)} real page images in {_real_dir}/")

predictions = {}
_page_seconds = []
for i, png_path in enumerate(_png_paths):
    # Printed BEFORE generate() starts, not just after -- a single page's generation can take
    # several minutes (especially the very first call in a session, when Qwen's Gated DeltaNet
    # kernels are still compiling -- see the training notebook's own note on this), and with no
    # pre-print there's no visible sign the cell is progressing rather than stuck for that whole
    # stretch.
    print(f"[{i + 1}/{len(_png_paths)}] generating {png_path.name}...")
    image = Image.open(png_path).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": _instruction}]}]
    input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")
    _start = time.monotonic()
    out = model.generate(**inputs, max_new_tokens=_MAX_NEW_TOKENS, use_cache=True, do_sample=False,
                          no_repeat_ngram_size=4)
    _elapsed = time.monotonic() - _start
    _page_seconds.append(_elapsed)
    _new_tokens = out.shape[1] - inputs["input_ids"].shape[1]
    _hit_cap = _new_tokens >= _MAX_NEW_TOKENS
    predictions[png_path.name] = processor.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"  done: {_elapsed:.1f}s, {_new_tokens} tokens"
          f"{' -- HIT CAP, possible truncation' if _hit_cap else ''}")

with open("real_predictions.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=1)
print(f"\nTotal: {sum(_page_seconds):.1f}s, mean {sum(_page_seconds) / len(_page_seconds):.1f}s/page")
print(f"Wrote real_predictions.json ({len(predictions)} pages) -- download it and score locally:")
print("  uv run python3 scripts/eval_finetune_real.py --predictions real_predictions.json")
